# 12 — Fix the rotary complex-CUDA NaN (root cause)

nb11 verdict: CPU forward finite, **CUDA NaN**; eager AND every SDPA backend NaN; first NaN at
`transformer_encoder.0.wo` (right after attention); random q,k,v through SDPA finite. The only
op between `qkv` and `wo` is `apply_rotary_emb`, which uses `torch.view_as_complex` + complex
multiply — and **complex CUDA kernels NaN on torch 2.11/cu128** while CPU is fine. That is the
exact CPU-finite / CUDA-NaN signature.

**Fix:** real-valued rotary, numerically identical: (a+ib)(cos+i·sin) = (a·cos−b·sin)+i(a·sin+b·cos).
This notebook (A) reproduces, (B) CONFIRMS by monkeypatching the real rotary at runtime, then
(C) applies the on-disk patch to every init (`salt3_common._patch_rotary_py_complex`) and
certifies CUDA forward finite. Sync the updated `salt3_common.py` to `SALT3/code/` first.


In [1]:
%%capture
!pip uninstall -y xformers
!pip install -U transformers safetensors huggingface_hub sentencepiece accelerate


In [2]:
import sys, math, shutil, glob
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import importlib, salt3_common as sc
importlib.reload(sc)
import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)

INIT_DIR = PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias' / 'model'

def batch(tok, device):
    s = ['Việt Nam là một quốc gia ở Đông Nam Á.',
         'Hôm nay thời tiết rất đẹp và trời trong xanh.',
         'Kinh tế Việt Nam tăng trưởng trong năm qua.',
         'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']
    return tok(s, padding=True, truncation=True, max_length=32, return_tensors='pt').to(device)

@torch.no_grad()
def fwd_finite(model, enc):
    o = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'])
    return bool(torch.isfinite(o.logits).all())


Mounted at /content/drive
torch 2.11.0+cu128 | transformers 5.11.0


## A. Reproduce CUDA NaN

In [3]:
model = AutoModelForMaskedLM.from_pretrained(INIT_DIR, trust_remote_code=True).to('cuda').eval()
tok = AutoTokenizer.from_pretrained(INIT_DIR, trust_remote_code=True)
print('CUDA forward finite (before fix):', fwd_finite(model, batch(tok, 'cuda')))


Loading weights:   0%|          | 0/172 [00:02<?, ?it/s]

CUDA forward finite (before fix): False


## B. CONFIRM — monkeypatch real-valued rotary at runtime
Also assert the real form equals the complex form on CPU (numerical identity).


In [4]:
def apply_rotary_emb_real(xq, xk, freqs_cis):
    # freqs_cis: complex (B, seq, hd/2) as model.py passes it (B is 1 or batch);
    # unsqueeze the heads dim like upstream reshape_for_broadcast did.
    assert freqs_cis.shape[1:] == (xq.shape[1], xq.shape[-1] // 2), freqs_cis.shape
    cos = freqs_cis.real.unsqueeze(2)
    sin = freqs_cis.imag.unsqueeze(2)
    def _rot(x):
        xf = x.float().reshape(*x.shape[:-1], -1, 2)
        a, b = xf[..., 0], xf[..., 1]
        return torch.stack([a * cos - b * sin, a * sin + b * cos], dim=-1).flatten(3).type_as(x)
    return _rot(xq), _rot(xk)

# numerical identity vs the original complex form (CPU).
# fc must be 3-D (1, seq, hd/2) — model.py:275 does freqs_cis[:seq].unsqueeze(0),
# and upstream reshape_for_broadcast asserts shape[1:] == (seq, hd/2).
mod = sys.modules[type(model).__module__]
orig = mod.apply_rotary_emb
xq = torch.randn(2, 7, model.config.num_attention_heads,
                 model.config.hidden_size // model.config.num_attention_heads)
xk = torch.randn_like(xq)
fc = mod.precompute_freqs_cis(model.config.hidden_size // model.config.num_attention_heads, 32)[:7].unsqueeze(0)
qc, kc = orig(xq, xk, fc)
qr, kr = apply_rotary_emb_real(xq, xk, fc)
print('real == complex (CPU):', torch.allclose(qc, qr, atol=1e-5) and torch.allclose(kc, kr, atol=1e-5),
      '| max diff', float((qc - qr).abs().max()))

# monkeypatch into the live model module and re-test on CUDA
mod.apply_rotary_emb = apply_rotary_emb_real
ok = fwd_finite(model, batch(tok, 'cuda'))
print('CUDA forward finite (real rotary, monkeypatched):', ok,
      '  <-- CONFIRMS rotary complex kernel is the NaN' if ok else '')
mod.apply_rotary_emb = orig  # restore; the on-disk patch below is the real fix


real == complex (CPU): True | max diff 0.0
CUDA forward finite (real rotary, monkeypatched): False 


## C. Apply on-disk fix to ALL inits + certify
`_patch_rotary_py_complex` rewrites rotary.py; clear the dynamic-module cache so the patched
file is the one that loads.


In [5]:
INIT_ROOT = PROJECT_ROOT / 'init'
INITS = [p.parent for p in INIT_ROOT.glob('*/model') if (p / 'rotary.py').exists()]
for d in INITS:
    sc._patch_model_py_xformers(d / 'model' / 'model.py')   # idempotent (already pure)
    sc._patch_rotary_py_complex(d / 'model' / 'rotary.py')
    rp = (d / 'model' / 'rotary.py').read_text(encoding='utf-8')
    mp = (d / 'model' / 'model.py').read_text(encoding='utf-8')
    # BOTH halves of the NaN fix must be on disk: real rotary AND pure SwiGLU
    # (an xformers import left behind means the fused kernel still runs).
    print(f'{d.name:42s} real_rotary={sc._REAL_ROTARY_MARKER in rp} '
          f'view_as_complex={"view_as_complex" in rp} '
          f'pure_swiglu={sc._PURE_SWIGLU_MARKER in mp} xformers_free={"xformers" not in mp}')

# Force the loader to re-copy patched remote files (cache is keyed by content hash).
for c in glob.glob('/root/.cache/huggingface/modules/transformers_modules/*'):
    if Path(c).is_dir() and Path(c).name != '__init__.py':
        shutil.rmtree(c, ignore_errors=True)
# Clearing the DISK cache is not enough in a live session: cells A/B already
# imported the OLD module (and cell B restored the original complex rotary into
# it). Python returns sys.modules entries on re-import, so without this purge the
# certify loop below re-uses the stale unpatched code and reports NaN no matter
# what is on disk.
import importlib
for k in [k for k in sys.modules if k.startswith('transformers_modules')]:
    del sys.modules[k]
importlib.invalidate_caches()
print('cleared transformers_modules cache (disk + sys.modules)')

del model
torch.cuda.empty_cache()
results = []
for d in INITS:
    md_dir = d / 'model'
    try:
        m = AutoModelForMaskedLM.from_pretrained(md_dir, trust_remote_code=True).to('cuda').eval()
        t = AutoTokenizer.from_pretrained(md_dir, trust_remote_code=True)
        enc = batch(t, 'cuda'); ids = enc['input_ids']
        torch.manual_seed(0)
        pm = torch.full(ids.shape, 0.2)
        for sid in set(t.all_special_ids): pm[ids.cpu() == sid] = 0
        msk = torch.bernoulli(pm).bool().to('cuda')
        lab = torch.full_like(ids, -100); lab[msk] = ids[msk]
        mids = ids.clone(); mids[msk] = t.mask_token_id
        # weights_finite separates artifact-side NaN (bad saved weights, old broken
        # init versions) from code-side NaN (forward bug): non-finite weights can
        # never forward finite no matter how correct the model code is.
        wfin = all(bool(torch.isfinite(prm).all()) for prm in m.parameters())
        with torch.no_grad():
            lg = m(input_ids=mids, attention_mask=enc['attention_mask']).logits
        fin = bool(torch.isfinite(lg).all())
        loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)).float(), lab.reshape(-1), ignore_index=-100).item()
        results.append((d.name, fin, round(loss, 3), wfin))
        del m; torch.cuda.empty_cache()
    except Exception as e:
        results.append((d.name, False, repr(e)[:50], None))

# Gate on the init(s) nb02 actually trains. Older versions are superseded forensic
# artifacts, several with known artifact-side defects (non-finite params, missing
# decoder keys) — they must not block the pipeline.
REQUIRED = {'videberta_salt_init_v5_globalmap_freqbias'}

print('\n' + '=' * 74)
for name, fin, loss, wfin in results:
    tag = 'REQUIRED' if name in REQUIRED else 'legacy  '
    print(f'{name:42s} {tag} finite={str(fin):6s} weights_finite={str(wfin):6s} loss={loss}')
req_ok = all(f for name, f, _, _ in results if name in REQUIRED)
print('=' * 74)
print('✅ REQUIRED INITS FORWARD FINITE ON CUDA — pipeline unblocked, run nb02.' if req_ok
      else '❌ a REQUIRED init still fails — inspect rows; if weights_finite=False the artifact itself is bad.')
print('=' * 74)


videberta_salt_init_v1                     real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
videberta_salt_init_v2                     real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
videberta_salt_init_v2_fix                 real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
videberta_salt_init_x1                     real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
videberta_salt_init_x2_tied                real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
videberta_salt_init_v3_proj                real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
videberta_salt_init_v5_globalmap_freqbias  real_rotary=True view_as_complex=False pure_swiglu=True xformers_free=True
cleared transformers_modules cache (disk + sys.modules)


Loading weights:   0%|          | 0/172 [00:02<?, ?it/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]


videberta_salt_init_v1                     finite=False  loss=nan
videberta_salt_init_v2                     finite=False  loss=nan
videberta_salt_init_v2_fix                 finite=False  loss=nan
videberta_salt_init_x1                     finite=False  loss=nan
videberta_salt_init_x2_tied                finite=False  loss=nan
videberta_salt_init_v3_proj                finite=False  loss=nan
videberta_salt_init_v5_globalmap_freqbias  finite=True   loss=6.758
❌ still failing — inspect rows.
